## Improving Reasoning with Inference-Time Scaling

In [1]:
import pathlib
import torch
import sympy
import tokenizers

In [2]:
device = torch.device("gpu" if torch.cuda.is_available() else "cpu")

In [3]:
from utils import load_model_tokenizer
model, tokenizer = load_model_tokenizer("base", device)

In [4]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def generate_text_stream_concat_flex(model, tokenizer, prompt, device, max_new_tokens,
                                    verbose=False, generate_func=None, **generate_kwargs):

    if generate_func is None: 
        generate_func = generate_text_basic_stream_cache
        
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

    generated_ids = []
    for token in generate_func(model=model, token_ids=input_ids, max_new_tokens=max_new_tokens,
                               eos_token_id=tokenizer.eos_token_id,**generate_kwargs):
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())

        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),end="",flush=True)
    return tokenizer.decode(generated_ids)

In [5]:
from utils import render_prompt
prompt = render_prompt(
        "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)

response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=1024, verbose=True,
    generate_func=generate_text_basic_stream_cache)

 \boxed{20}

Wrong answer

### CoT prompting

In [6]:
prompt_cot = prompt + " \n\nExplain step by step."

response_cot = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True)

 To solve the problem, we need to find the value of \( x \) such that half the value of \( 3x - 9 \) is equal to \( x + 37 \).

### Step 1: Set up the equation
We are given that half the value of \( 3x - 9 \) is equal to \( x + 37 \). This can be written as:
\[
\frac{1}{2}(3x - 9) = x + 37
\]

### Step 2: Eliminate the fraction
To eliminate the fraction, multiply both sides of the equation by 2:
\[
2 \cdot \frac{1}{2}(3x - 9) = 2(x + 37)
\]
Simplifying both sides:
\[
3x - 9 = 2x + 74
\]

### Step 3: Solve for \( x \)
Subtract \( 2x \) from both sides to isolate \( x \):
\[
3x - 2x - 9 = 74
\]
Simplify:
\[
x - 9 = 74
\]
Add 9 to both sides to solve for \( x \):
\[
x = 74 + 9
\]
\[
x = 83
\]

### Final Answer:
\[
\boxed{83}
\]

In [7]:
from IPython.display import Latex, display
display(Latex(response_cot))

<IPython.core.display.Latex object>

### Temperature Scaling

In [8]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_temp_stream_cache(model, token_ids, max_new_tokens, eos_token_id=None, temperature=0.0):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1, :] #batch, seqlen, emb_dim
    for _ in range(max_new_tokens):
        orig_device = token_ids.device
        
        if temperature <= 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)

        else:
            logits = out/temperature
            probas = torch.softmax(logits, dim = -1)
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1, :]

In [9]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_temp_stream_cache,
    temperature=1.1
)

 We begin with the equation \(\frac{1}{2}(3x - 9) = x + 37\).
First, distribute the \(\frac{1}{2}\) on the left-side of the equation to get \(3x - 9 = 2(x + 37)\).
Then simplify the right side of the equation by using the distributive property and combining like terms: \(3x - 9 = 2x + 74\).
Next, subtract \(2x\) from both sides of the equation to get \(x - 9 = 2x + 74\).
Next, subtract \(x\) from both sides of the equation to reduce the number of \(x\) terms on the right side to exactly 1: \(-9 = x + 74\).
Finally, subtract 74 from both sides of the equation to get \(x = -83\).
The value of \(x\) is \(\boxed{-83}\).

### Top-p Sampling

In [10]:
def top_p_filter(prob, p):
    if p<0 or p>1:
        return prob
    
    sorted_prob, sorted_indices = torch.sort(prob, dim=-1, descending=True)
    cumprob = torch.cumsum(sorted_prob, dim=-1)
    prefix = cumprob - sorted_prob
    keep = prefix < p #boolean array of what to keep
    keep[:, 0] = True #keep first element minimum override
    kept_sorted = torch.where(keep, sorted_prob, torch.zeros_like(sorted_prob))
    filtered = torch.zeros_like(prob).scatter(1, sorted_indices, kept_sorted)
    denom = torch.sum(filtered, dim = -1, keepdim=True).clamp_min(1e-12) #renormalizing
    return filtered/denom

In [11]:
dummy = tokenizer.encode("hi my name is sanchit")

In [12]:
dummy = torch.tensor(dummy).unsqueeze(0)

In [17]:
with torch.inference_mode():
    next_token_logits = model(dummy)
print(f"model output shape: {next_token_logits.shape}")
print(f"last token: {next_token_logits[:, -1, :].shape}")

model output shape: torch.Size([1, 7, 151936])
last token: torch.Size([1, 151936])


In [31]:
@torch.inference_mode
def generate_text_top_p_stream_cache(model, token_ids, max_new_tokens, eos_token_id = None, temperature = 0.0, top_p = None):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    
    logits = model(token_ids, cache)[:, -1, :]
    
    for _ in range(max_new_tokens):
        orig_device = token_ids.device

        if temperature == 1.0 or temperature is None :
            next_token = torch.argmax(logits, dim = -1, keepdim = True) 

        else:
            logits = logits/temperature #scale logits
            probs = torch.softmax(logits, dim = -1) # convert to prob
            probs = top_p_filter(probs, top_p) # take top p
            next_token = torch.multinomial(probs.cpu(), num_samples=1) #choose 
            next_token = next_token.to(orig_device)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token
        logits = model(next_token, cache=cache)[:, -1, :]

In [32]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=1024, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.5,top_p=0.8 
)

 \boxed{18}

### Self-consistency

In [33]:
from reasoning_from_scratch.ch03 import extract_final_candidate
from collections import Counter

def self_consistency_vote(
    model, tokenizer, prompt, device,
    num_samples=10, temperature=0.8, top_p=0.9, max_new_tokens=2048,
    show_progress=True, show_long_answer=False, seed=None,
):
    full_answers, short_answers = [], []

    # 1) Sample multiple answers
    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model, tokenizer=tokenizer, prompt=prompt, device=device,
            max_new_tokens=max_new_tokens, verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature, top_p=top_p,
        )

        # 2) Extract the final (short) answer from each answer
        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    # 3) Choose the most frequent final answer (self-consistency vote)
    counts = Counter(short_answers)
    groups = {s: [] for s in counts}
    for idx, s in enumerate(short_answers):
        groups[s].append(idx)

    mc = counts.most_common()
    if not mc:
        majority_winners, final_answer = [], None
    else:
        top_freq = mc[0][1]
        majority_winners = [s for s, f in mc if f == top_freq]
        final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

In [34]:
results = self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

[Sample 1/5] → '83'
[Sample 2/5] → '22'
[Sample 3/5] → '54'
[Sample 4/5] → '83'
[Sample 5/5] → '83'
